<a href="https://colab.research.google.com/github/shivansh2310/The-elements-of-quantitative-investing-/blob/main/The_Friction_of_Reality_%26_Hedging_(Chapters_11%E2%80%9312).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The Cost of Doing Business

* Linear Costs: Bid-ask spreads and broker fees. These scale steadily with your trade size.

* Quadratic Costs (Market Impact): The real enemy. If you try to execute that massive Risk Parity weight in a less liquid stock, your own buying pressure moves the price against you. In institutional optimization, you must add a quadratic penalty to the objective function (using the Frobenius norm) to stop the math from demanding impossible, market-moving trades.

### B. The Factor-Mimicking Portfolio (FMP)

This is the crown jewel of quantitative hedging.If your fundamental model from Day 3 says "Value is going to outperform," you don't just buy the cheapest stock. That stock has Market Beta, Sector Risk, and idiosyncratic garbage attached to it.

An FMP is a mathematically engineered long/short portfolio designed to have exactly $1.0$ exposure to your target factor (Value) and exactly $0.0$ exposure to everything else (like the broader market).

The exact FMP matrix equation to isolate this alpha is:$$h = \Sigma^{-1} X (X^T \Sigma^{-1} X)^{-1} e_k$$

### Engineering the Pure Value FMP

In [20]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf

In [21]:
stock_data = {
    "MSFT": "Tech", "NVDA": "Tech", "AAPL": "Tech",
    "NEE": "Utilities", "AEP": "Utilities", "VST": "Utilities",
    "CAT": "Industrials", "GE": "Industrials", "PWR": "Industrials",
    "JPM": "Financials", "GS": "Financials", "V": "Financials",
    "XOM": "Energy", "CVX": "Energy",
    "META": "Comm", "GOOGL": "Comm",
    "LLY": "Healthcare", "UNH": "Healthcare",
    "AMZN": "Consumer", "WMT": "Consumer"
}
tickers = list(stock_data.keys())

print("Fetching 2-year history for Covariance calculation...")
# 2. Get Historical Returns for the 20 Stocks
# We need history to calculate how they move together (Covariance)
hist_prices = yf.download(tickers, period="2y")['Close']
hist_returns = hist_prices.pct_change().dropna()

# Ensure the columns in the returns matrix exactly match the order of our Day 3 dataframe
hist_returns = hist_returns[tickers]

Fetching 2-year history for Covariance calculation...


[*********************100%***********************]  20 of 20 completed


In [22]:
hist_returns

Ticker,MSFT,NVDA,AAPL,NEE,AEP,VST,CAT,GE,PWR,JPM,GS,V,XOM,CVX,META,GOOGL,LLY,UNH,AMZN,WMT
Date,,,,,,,,,,,,,,,,,,,,
2024-06-11,0.011242,-0.007144,0.072649,-0.054956,-0.001581,0.034006,-0.006978,-0.015229,-0.000037,-0.026301,-0.020461,-0.001345,-0.008048,-0.001595,0.009690,0.009199,0.000948,0.002465,0.000909,-0.003435
2024-06-12,0.019368,0.035481,0.028578,-0.006599,-0.005090,-0.007235,0.004339,0.004303,0.019259,-0.014561,0.009971,-0.015837,-0.011055,-0.014506,0.002700,0.006624,0.001709,-0.006348,-0.001816,-0.006294
2024-06-13,0.001179,0.035224,0.005491,0.012732,-0.000682,-0.023852,-0.007210,-0.033282,-0.000325,0.011121,-0.005237,0.003218,-0.008023,-0.008948,-0.009315,-0.014793,0.018483,0.008579,-0.016373,0.005882
2024-06-14,0.002242,0.017514,-0.008168,-0.001640,0.001137,-0.009163,-0.014983,0.001220,-0.012324,0.000620,0.000246,-0.001954,-0.008452,-0.001766,0.001112,0.009306,-0.005525,-0.000362,-0.000925,0.004797
2024-06-17,0.013105,-0.006824,0.019672,-0.010265,-0.002841,-0.040530,0.002893,0.047151,0.012038,0.006193,0.008332,0.001884,-0.006874,0.004064,0.004899,0.002545,0.007468,-0.011696,0.002178,0.005969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-03,-0.031656,-0.036218,-0.015673,-0.012838,-0.006294,-0.026397,0.017993,-0.009694,0.013611,-0.000365,-0.022131,-0.015505,0.019858,0.011517,0.042418,-0.007904,0.013748,-0.002434,-0.025339,0.033876
2026-06-04,0.001661,0.019394,0.003126,0.013005,0.011717,-0.000650,0.015440,0.041349,0.004891,0.033372,0.049557,0.024904,-0.003213,-0.007169,0.007368,0.036770,0.043095,0.051645,0.015079,0.007272
2026-06-05,-0.026586,-0.062014,-0.012499,0.009206,0.010564,-0.032141,-0.038491,0.001068,-0.033455,0.004760,-0.049359,0.010588,-0.013944,-0.005522,-0.055085,-0.009834,0.005465,0.007567,-0.030576,0.009682


In [24]:
# Calculate the Shrunk, Annualized Covariance Matrix (The Day 5 Fix)
shrunk_cov_daily = LedoitWolf().fit(hist_returns).covariance_
cov_annual = shrunk_cov_daily * 252

In [25]:
fund_data = []
for ticker in tickers:
    try:
        tk = yf.Ticker(ticker)
        pe = tk.info.get('trailingPE', np.nan)
        # Convert P/E to Earnings Yield (Value Factor)
        # If P/E is missing or negative, we temporarily set to NaN
        ey = 1 / pe if pd.notna(pe) and pe > 0 else np.nan
        fund_data.append({"Ticker": ticker, "EarningsYield": ey})
    except Exception:
        fund_data.append({"Ticker": ticker, "EarningsYield": np.nan})

df = pd.DataFrame(fund_data).set_index("Ticker")

# Impute missing data with the mean so the matrix math doesn't fail
df['EarningsYield'] = df['EarningsYield'].fillna(df['EarningsYield'].mean())

In [26]:
# Z-Score the Value exposure (Mean 0, StdDev 1)
df['Value_Z'] = (df['EarningsYield'] - df['EarningsYield'].mean()) / df['EarningsYield'].std()

# Build Exposure Matrix (X): Add a constant (Market Beta intercept)
X_fmp = sm.add_constant(df[['Value_Z']])
X_matrix = X_fmp.values

In [27]:
Sigma = cov_annual
Sigma_inv = np.linalg.inv(Sigma)

# The Core FMP Equation: h = Sigma^-1 * X * (X^T * Sigma^-1 * X)^-1
XT_SigmaInv_X = X_matrix.T @ Sigma_inv @ X_matrix
XT_SigmaInv_X_inv = np.linalg.inv(XT_SigmaInv_X)

H_matrix = Sigma_inv @ X_matrix @ XT_SigmaInv_X_inv

# Extract the Pure Value Factor (Index 1 is 'Value_Z', Index 0 is the Constant/Beta)
value_fmp_weights = H_matrix[:, 1]

In [28]:
fmp_df = pd.Series(value_fmp_weights, index=tickers).sort_values(ascending=False)

print("\n" + "="*50)
print(" PURE VALUE FACTOR-MIMICKING PORTFOLIO (FMP)")
print("="*50)
print(fmp_df.round(4))
print("-" * 50)
print(f"Net Market Exposure (Sum of weights): {fmp_df.sum():.6f}")
print("="*50)


 PURE VALUE FACTOR-MIMICKING PORTFOLIO (FMP)
JPM      0.1794
AEP      0.0900
XOM      0.0385
META     0.0373
NEE      0.0301
GS       0.0261
GOOGL    0.0201
VST      0.0172
NVDA     0.0079
MSFT     0.0071
UNH     -0.0103
LLY     -0.0211
AAPL    -0.0261
AMZN    -0.0275
GE      -0.0505
PWR     -0.0547
WMT     -0.0577
CAT     -0.0582
V       -0.0657
CVX     -0.0819
dtype: float64
--------------------------------------------------
Net Market Exposure (Sum of weights): 0.000000
